# 03 - Avaliacao e Teste do Modelo SARIMAX

## Estrutura

### Parte A — Avaliacao do modelo mensal
- Walk-forward cross-validation (janelas deslizantes)
- Degradacao da precisao por horizonte (1, 2, 3, 4 meses)
- Testes estatisticos de residuos (Ljung-Box, Shapiro-Wilk)
- Intervalo de confianca 95% nas previsoes

### Parte B — Teste com janelas diarias (7, 14, 30 dias)
- SARIMAX diario com sazonalidade semanal (s=7)
- Avaliacao por janela: 7, 14 e 30 dias
- Heatmap e comparacao visual das janelas

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.sarimax_model import (
    load_data, get_top_products,
    prepare_monthly_data, build_exogenous_features,
    compute_feature_importance, weight_features,
    select_sarimax_order, fit_sarimax, suggest_d,
    PLOTS_DIR, METRICS_DIR,
)
from src.models.evaluation import (
    walk_forward_cv,
    evaluate_monthly_horizons,
    residual_tests,
    confidence_interval_plot,
    plot_cv_summary,
    generate_evaluation_report,
    EVAL_PLOTS_DIR,
)
from src.models.test_model import (
    prepare_daily_series,
    run_daily_sarimax,
    run_daily_pipeline,
    plot_daily_forecast,
    plot_window_comparison,
    plot_smape_heatmap,
    TEST_PLOTS_DIR,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

In [ ]:
DATA_PATH = os.path.join(project_root, 'data', 'processed', 'contas_a_receber_discretizado.csv')
TOP_N = 10
TEST_MONTHS = 4

df = load_data(DATA_PATH)
top_products = get_top_products(df, TOP_N)
print(f'Dados carregados: {len(df):,} registros')
print(f'Produtos: {top_products}')

---
## Parte A — Avaliacao do modelo mensal


### A.1 Walk-forward Cross-Validation

A validacao walk-forward expande o conjunto de treino a cada fold,
simulando como o modelo se comportaria em producao ao longo do tempo.

```
Fold 1: [====treino====] [teste]
Fold 2: [======treino======] [teste]
Fold 3: [========treino========] [teste]
```

In [ ]:
# Walk-forward CV para o produto mais vendido (horizonte 1 mes)
ref_prod = top_products[0]
print(f'Walk-forward CV — produto {ref_prod} (horizonte = 1 mes)\n')

cv_df = walk_forward_cv(df, ref_prod, n_splits=5, horizon=1, min_train=12, verbose=True)
print(f'\nResultados por fold:')
display(cv_df.round(3))
print(f'\nMedia dos folds: SMAPE={cv_df["SMAPE_%"].mean():.1f}% | R2={cv_df["R2"].mean():.3f}')

### A.2 Degradacao por horizonte (1, 2, 3, 4 meses)

Avalia como a precisao cai conforme o horizonte de previsao aumenta.

In [ ]:
print('Avaliando degradacao por horizonte para todos os produtos...')
print('(pode levar alguns minutos)\n')

cv_all = {}
for prod in top_products:
    print(f'Produto {prod}:')
    try:
        df_h = evaluate_monthly_horizons(df, prod, horizons=[1, 2, 3, 4],
                                          n_splits=4, verbose=True)
        cv_all[prod] = df_h
    except Exception as e:
        print(f'  Erro: {e}')
        cv_all[prod] = pd.DataFrame()

print('\nConcluido!')

In [ ]:
# Tabela resumo por produto e horizonte
rows = []
for prod, df_h in cv_all.items():
    if not df_h.empty:
        rows.append(df_h)

if rows:
    horizon_summary = pd.concat(rows, ignore_index=True)
    pivot = horizon_summary.pivot_table(
        index='produto_cod', columns='horizonte_meses',
        values='SMAPE_%_medio', aggfunc='mean'
    )
    pivot.columns = [f'H{c}m' for c in pivot.columns]
    print('SMAPE medio por produto e horizonte:')
    display(pivot.round(1))

# Grafico de degradacao
plot_cv_summary(cv_all, EVAL_PLOTS_DIR)

### A.3 Testes estatisticos de residuos

| Teste | H0 | Interpretacao |
|---|---|---|
| **Ljung-Box** | Residuos sao independentes | p > 0.05 -> modelo adequado |
| **Shapiro-Wilk** | Residuos sao normais | p > 0.05 -> normalidade |
| **Jarque-Bera** | Assimetria e curtose normais | p > 0.05 -> distribuicao OK |

In [ ]:
# Re-treina modelos para obter resultados com todos os dados de treino
print('Treinando modelos e testando residuos...\n')

fitted_models = {}
train_series = {}
test_series = {}
exog_test_weighted = {}
use_log_map = {}

for prod in top_products:
    try:
        ts = prepare_monthly_data(df, prod)
        exog = build_exogenous_features(df, prod)
        exog = exog.reindex(ts.index).ffill().fillna(0)
        common = ts.index.intersection(exog.index)
        ts, exog = ts.loc[common], exog.loc[common]

        cv = ts.std() / (ts.mean() + 1e-9)
        use_log = cv > 0.7
        ts_model = np.log1p(ts) if use_log else ts.copy()

        cut = len(ts) - TEST_MONTHS
        ts_tr = ts_model.iloc[:cut]
        exog_tr = exog.iloc[:cut]
        exog_te = exog.iloc[cut:]

        imp = compute_feature_importance(exog_tr, ts_tr)
        exog_tr_w = weight_features(exog_tr, imp)
        exog_te_w = weight_features(exog_te, imp)

        d = suggest_d(ts_tr)
        order, seasonal_order = select_sarimax_order(
            ts_tr, exog_tr_w, d=d, D=0, m=12,
            p_range=range(0, 3), q_range=range(0, 3),
            P_range=range(0, 2), Q_range=range(0, 2),
        )
        result = fit_sarimax(ts_tr, exog_tr_w, order, seasonal_order)

        fitted_models[prod] = result
        train_series[prod] = ts.iloc[:cut]
        test_series[prod] = ts.iloc[cut:]
        exog_test_weighted[prod] = exog_te_w
        use_log_map[prod] = use_log
        print(f'  {prod}: OK')
    except Exception as e:
        print(f'  {prod}: Erro - {e}')

In [ ]:
# Testes de residuos para cada produto
residual_rows = []
for prod in top_products:
    if prod in fitted_models:
        tests = residual_tests(fitted_models[prod], prod, verbose=True)
        residual_rows.append(tests)

residual_df = pd.DataFrame(residual_rows)
print('\n=== Tabela de testes de residuos ===')
cols = ['produto_cod', 'ljung_box_p', 'ljung_box_ok', 'shapiro_p', 'shapiro_normal',
        'resid_mean', 'resid_std', 'resid_skew']
display(residual_df[cols].round(4))

### A.4 Intervalo de confianca 95%

In [ ]:
# IC para os 3 melhores produtos (menor SMAPE)
try:
    summary_monthly = pd.read_csv(
        os.path.join(project_root, 'results', 'metrics', 'metricas_sarimax.csv'),
        sep=';'
    )
    best_prods = summary_monthly.nsmallest(3, 'SMAPE_%')['produto_cod'].astype(str).tolist()
except Exception:
    best_prods = [str(top_products[0])]

print(f'Gerando graficos de IC para: {best_prods}')
for prod in best_prods:
    if prod in fitted_models:
        confidence_interval_plot(
            fitted_models[prod],
            train_series[prod],
            test_series[prod],
            exog_test_weighted[prod],
            prod,
            use_log=use_log_map.get(prod, False),
            save_dir=EVAL_PLOTS_DIR,
        )

---
## Parte B — Teste com janelas diarias (7, 14, 30 dias)

O modelo mensal serve para planejamento de compras (1-4 meses).
O modelo **diario** com sazonalidade semanal (s=7) serve para gestao de estoque
no curto prazo: quanto pedir para a proxima semana, quinzena ou mes.

**Estrategia de teste:**
- Treina com toda a serie menos os ultimos 30 dias
- Preve 7, 14 e 30 dias a frente
- Avalia MAE, RMSE, SMAPE, R2 para cada janela

### B.1 Analise exploratoria da demanda diaria

In [ ]:
# Estatisticas diarias por produto
print('Estatisticas das series diarias:\n')
daily_stats = []
for prod in top_products:
    ts = prepare_daily_series(df, prod)
    cv = ts.std() / (ts.mean() + 1e-9)
    zeros = (ts == 0).sum()
    daily_stats.append({
        'produto': str(prod),
        'n_dias': len(ts),
        'media': round(ts.mean(), 1),
        'std': round(ts.std(), 1),
        'CV': round(cv, 2),
        'zeros': zeros,
        'max': ts.max(),
        'log_transform': cv > 1.0,
    })

stats_df = pd.DataFrame(daily_stats)
display(stats_df)

In [ ]:
# Visualiza series diarias do produto de referencia
ref_prod = top_products[0]
ts_daily = prepare_daily_series(df, ref_prod)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Serie completa
axes[0].plot(ts_daily.index, ts_daily.values, linewidth=0.8, color='#1f77b4')
axes[0].set_title(f'Demanda diaria — Produto {ref_prod}', fontsize=11)
axes[0].set_ylabel('Qtd')
axes[0].grid(True, alpha=0.3)

# Ultimos 90 dias
axes[1].plot(ts_daily.iloc[-90:].index, ts_daily.iloc[-90:].values,
             linewidth=1.2, color='#1f77b4', marker='o', markersize=3)
axes[1].axvline(ts_daily.index[-30], color='red', linestyle='--',
                linewidth=1, label='Inicio janela 30d')
axes[1].axvline(ts_daily.index[-14], color='orange', linestyle='--',
                linewidth=1, label='Inicio janela 14d')
axes[1].axvline(ts_daily.index[-7], color='green', linestyle='--',
                linewidth=1, label='Inicio janela 7d')
axes[1].set_title('Ultimos 90 dias com janelas de teste', fontsize=11)
axes[1].set_ylabel('Qtd')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(TEST_PLOTS_DIR / f'serie_diaria_{ref_prod}.png', dpi=150, bbox_inches='tight')
plt.show()

### B.2 Pipeline diario — todos os produtos

In [ ]:
# Filtra produtos com dados suficientes (>= 90 dias)
valid_prods = []
for prod in top_products:
    ts = prepare_daily_series(df, prod)
    if len(ts) >= 90:
        valid_prods.append(prod)
    else:
        print(f'  Produto {prod}: apenas {len(ts)} dias — ignorado')

print(f'\nProdutos com dados suficientes: {valid_prods}')
print('Iniciando pipeline diario...\n')

In [ ]:
daily_summary = run_daily_pipeline(
    df,
    valid_prods,
    horizons=[7, 14, 30],
    min_obs=90,
    verbose=True,
)

### B.3 Resultados — metricas por janela

In [ ]:
print('=== METRICAS DIARIAS POR JANELA ===')
display(daily_summary[['produto_cod','horizonte_dias','MAE','RMSE','SMAPE_%','R2']].round(2))

print('\n=== SMAPE MEDIO POR JANELA ===')
print(daily_summary.groupby('horizonte_dias')['SMAPE_%'].mean().round(1).to_string())

print('\n=== INTERPRETACAO ===')
for h in [7, 14, 30]:
    sub = daily_summary[daily_summary['horizonte_dias'] == h]
    if not sub.empty:
        smape = sub['SMAPE_%'].mean()
        bons = (sub['SMAPE_%'] < 30).sum()
        print(f'Janela {h:2d} dias: SMAPE medio={smape:.1f}% | {bons}/{len(sub)} produtos com SMAPE<30%')

In [ ]:
# Pivot table SMAPE por produto x janela
if not daily_summary.empty:
    pivot = daily_summary.pivot_table(
        index='produto_cod', columns='horizonte_dias',
        values='SMAPE_%', aggfunc='mean'
    )
    pivot.columns = [f'{c} dias' for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(8, 5))
    import seaborn as sns
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
                vmin=0, vmax=100, ax=ax, linewidths=0.5,
                cbar_kws={'label': 'SMAPE (%)'})
    ax.set_title('SMAPE (%) por produto e janela de previsao')
    plt.tight_layout()
    plt.savefig(TEST_PLOTS_DIR / 'heatmap_smape_janelas.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Conclusao

### Quando usar cada modelo

| Necessidade | Modelo | Arquivo |
|---|---|---|
| Previsao de compras (1-4 meses) | SARIMAX mensal | `02_model_training.ipynb` |
| Reposicao semanal (7 dias) | SARIMAX diario | `03_evaluation.ipynb` |
| Planejamento quinzenal (14 dias) | SARIMAX diario | `03_evaluation.ipynb` |
| Gestao mensal de estoque (30 dias) | SARIMAX diario | `03_evaluation.ipynb` |

### Guia de interpretacao SMAPE
| SMAPE | Classificacao |
|---|---|
| < 20% | Excelente |
| 20-30% | Bom |
| 30-60% | Aceitavel |
| > 60% | Ruim (serie irregular) |

### Arquivos gerados
- `results/metrics/cv_por_horizonte.csv` — degradacao mensal
- `results/metrics/residuos_testes.csv` — testes de residuos
- `results/metrics/metricas_diarias_janelas.csv` — metricas diarias 7/14/30d
- `results/plots/evaluation/` — IC 95% e graficos de CV
- `results/plots/test/` — previsoes diarias e comparativo de janelas